# RazorStitch — Benchmark & Export Policy

Compare DQN vs baselines on held-out simulator envs, then export `policy_rules.json` for the Vercel API (no PyTorch on server).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "packages").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

SEED = 42
CKPT = ROOT / f"eval/checkpoints/dqn_train_seed{SEED}.pt"

In [ ]:
from packages.policy.train import run_benchmark

results = run_benchmark(
    seeds=[42, 1337, 2025],
    episodes_per_seed=500,
    checkpoint=CKPT,
    out_dir=ROOT / "eval/results",
)
print("Wrote eval/results/benchmark.json and tables.md")

In [ ]:
import json
import pandas as pd

rows = []
for r in results["baselines"]:
    rows.append({"policy": r["policy"], "seed": r["seed"], **{k: r[k] for k in ["recovery_rate", "gross_recovered_value_inr", "duplicate_incidents"]}})
for r in results.get("dqn", []):
    rows.append({"policy": "dqn", "seed": r["seed"], **{k: r[k] for k in ["recovery_rate", "gross_recovered_value_inr", "duplicate_incidents"]}})

df = pd.DataFrame(rows)
summary = df.groupby("policy").agg(
    recovery_rate=("recovery_rate", "mean"),
    gross_inr=("gross_recovered_value_inr", "sum"),
    duplicates=("duplicate_incidents", "sum"),
).sort_values("recovery_rate", ascending=False)
summary

In [ ]:
from IPython.display import Markdown

tables = (ROOT / "eval/results/tables.md").read_text()
Markdown(tables)

In [ ]:
from packages.policy.dqn import DQNAgent
from packages.policy.export_rules import export_reason_preferences

agent = DQNAgent.load(CKPT)
rules_path = ROOT / "eval/checkpoints/policy_rules.json"
prefs = export_reason_preferences(agent, rules_path)
print(json.dumps(prefs["reasons"]["insufficient_funds"], indent=2))
print("\nExported:", rules_path)

## Next step

Copy `eval/checkpoints/policy_rules.json` into `apps/web/data/` for the policy recommend API.